# Agentic Vision Cloud Orchestrator — Interactive Demo

This notebook demonstrates the complete agentic pipeline end-to-end:

| Step | Component | Technology |
|------|-----------|-----------|
| 1 | Pet breed identification from image URL | ResNet50 CV service (Docker) |
| 2 | Breed knowledge lookup | In-memory DB / Pinecone vector DB |
| 3 | Web search for live veterinary info | DuckDuckGo via `ddgs` |
| 4 | Multi-turn reasoning & orchestration | LangGraph (ReAct pattern) |
| 5 | Monitoring | Prometheus + Grafana |

---

> **Prerequisites**: `cv_service` container running on `localhost:8000`  
> Start it with: `docker-compose up cv_service`


## 1 · Environment setup

In [1]:
import sys, os

# Add project root to path so agent/* modules are importable from the notebook.
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

# Verify required keys are present (values are hidden).
required = ["OPENAI_API_KEY"]
optional  = ["PINECONE_API_KEY", "CV_SERVICE_URL"]

for key in required:
    status = "✅" if os.getenv(key) else "❌ MISSING"
    print(f"{key}: {status}")

for key in optional:
    status = "✅" if os.getenv(key) else "⚠️  not set (optional)"
    print(f"{key}: {status}")

cv_url = os.getenv("CV_SERVICE_URL", "http://localhost:8000")
print(f"\nCV service URL: {cv_url}")


OPENAI_API_KEY: ✅
PINECONE_API_KEY: ✅
CV_SERVICE_URL: ✅

CV service URL: http://localhost:8000


## 2 · CV service health check

In [2]:
import httpx

cv_url = os.getenv("CV_SERVICE_URL", "http://localhost:8000")

try:
    r = httpx.get(f"{cv_url}/health", timeout=5)
    print(f"Status: {r.status_code}")
    print(r.json())
except Exception as e:
    print(f"⚠️  CV service not reachable: {e}")
    print("Run: docker-compose up cv_service")


Status: 200
{'status': 'ok', 'model_loaded': True}


## 3 · Direct CV tool call

Call `cv_predict` directly (without the agent) to verify the model pipeline.


In [3]:
from agent.tools.cv_tool import cv_predict
import json

TEST_IMAGES = {
    "Beagle":  "https://images.dog.ceo/breeds/beagle/n02088364_10108.jpg",
    "Poodle":  "https://images.dog.ceo/breeds/poodle-standard/n02113799_2280.jpg",
    "Labrador": "https://images.dog.ceo/breeds/labrador/n02099712_4323.jpg",
}

for label, url in TEST_IMAGES.items():
    result = json.loads(cv_predict.invoke({"image_url": url}))
    if "error" in result:
        print(f"[{label}] ERROR: {result['error']}")
    else:
        top1 = result["breed"]
        conf  = result["confidence"]
        match = "✅" if label.split()[0].lower() in top1.lower() else "⚠️ "
        print(f"{match} [{label}]  →  {top1}  (conf: {conf:.2%})")
        print(f"   Top-5: {[x['breed'] for x in result['top5']]}")
    print()


✅ [Beagle]  →  Beagle  (conf: 83.42%)
   Top-5: ['Beagle', 'Chihuahua', 'American Pit Bull Terrier', 'Saint Bernard', 'Basset Hound']

⚠️  [Poodle]  →  Bengal  (conf: 37.88%)
   Top-5: ['Bengal', 'English Cocker Spaniel', 'American Pit Bull Terrier', 'Miniature Pinscher', 'Egyptian Mau']

⚠️  [Labrador]  →  American Pit Bull Terrier  (conf: 22.55%)
   Top-5: ['American Pit Bull Terrier', 'German Shorthaired', 'Beagle', 'Great Pyrenees', 'American Bulldog']



## 4 · Breed knowledge base lookup

`db_query` provides structured, factual data for all 37 Oxford Pets breeds
without any external service call.


In [4]:
from agent.tools.db_tool import db_query, _BREED_DB
import json

# Show a sample of the knowledge base
sample_breeds = ["beagle", "siamese", "maine_coon", "pug"]

for breed in sample_breeds:
    data = json.loads(db_query.invoke({"breed_name": breed}))
    print(f"── {breed.upper()} ──────────────────────────")
    for k, v in data.items():
        print(f"  {k:15s}: {v}")
    print()

print(f"Total breeds in DB: {len(_BREED_DB)}")


── BEAGLE ──────────────────────────
  breed          : beagle
  type           : dog
  description    : Compact scent hound with floppy ears and a merry temperament.
  origin         : UK
  temperament    : Curious, friendly, merry, determined
  health_notes   : Epilepsy, hip dysplasia, hypothyroidism, ear infections.
  lifespan       : 10-15 years
  size           : small-medium

── SIAMESE ──────────────────────────
  breed          : siamese
  type           : cat
  description    : Sleek colour-pointed cat with striking blue eyes.
  origin         : Thailand
  temperament    : Vocal, social, demanding, highly intelligent
  health_notes   : Prone to amyloidosis, dental disease, and respiratory issues.
  lifespan       : 12-20 years
  size           : medium

── MAINE_COON ──────────────────────────
  breed          : maine_coon
  type           : cat
  description    : Large, tufted-eared cat with shaggy semi-long coat.
  origin         : USA (Maine)
  temperament    : Dog-like, pl

## 5 · Single-turn agent invocation

`run_agent` executes the full LangGraph pipeline for one user message.
The graph runs the ReAct loop: **agent → tools → extract_breed → agent → …**


In [5]:
from agent.runner import run_agent
from agent.tools import ALL_TOOLS

IMAGE_URL = "https://images.dog.ceo/breeds/beagle/n02088364_10108.jpg"

result = run_agent(
    user_input=f"What breed is the dog in this image and what are its main health issues? {IMAGE_URL}",
    tools=ALL_TOOLS,
)

result_single = result

print("=== Agent reply ===")
print(result["messages"][-1].content)
print()
print(f"Breed identified : {result['breed_identified']}")
print(f"Total turns      : {result['turn_count']}")
print(f"Messages in state: {len(result['messages'])}")


=== Agent reply ===
The dog in the image is identified as a **Beagle** with a confidence score of 83.42%. 

### Beagle Overview:
- **Description**: Compact scent hound with floppy ears and a merry temperament.
- **Origin**: United Kingdom
- **Temperament**: Curious, friendly, merry, determined
- **Size**: Small to medium
- **Lifespan**: 10-15 years

### Main Health Issues:
Beagles are prone to several health issues, including:
- **Epilepsy**
- **Hip Dysplasia**
- **Hypothyroidism**
- **Ear Infections**

These conditions are important to monitor for maintaining the health and well-being of a Beagle.

Breed identified : Beagle
Total turns      : 2
Messages in state: 6


## 6 · Multi-turn conversation

`AgentSession` preserves history across turns so the agent remembers
what breed was identified and can answer follow-up questions without
repeating the CV call.


In [6]:
from agent.runner import AgentSession
from agent.tools import ALL_TOOLS

session = AgentSession(tools=ALL_TOOLS)
session_ref = session

print(f"Session ID: {session.session_id}\n")
print("=" * 60)

# Turn 1 — image identification
IMAGE_URL = "https://images.dog.ceo/breeds/beagle/n02088364_10108.jpg"
reply1 = session.chat(f"Identify the breed in this image: {IMAGE_URL}")

print("=" * 60)

# Turn 2 — follow-up (no image URL needed, agent remembers context)
reply2 = session.chat("What is the typical lifespan and size of this breed?")

print("=" * 60)

# Turn 3 — veterinary follow-up
reply3 = session.chat("Are there any specific diet recommendations for this breed?")

print("=" * 60)
print(f"\nBreed identified: {session.breed_identified}")
print(f"Total messages  : {len(session.history)}")


Session ID: c3714833-0f15-42ff-8b8a-db26afd7359b


[Agent]: The breed identified in the image is a **Beagle** with a confidence score of 83.42%. 

### Beagle Overview:
- **Type**: Dog
- **Description**: Compact scent hound with floppy ears and a merry temperament.
- **Origin**: United Kingdom
- **Temperament**: Curious, friendly, merry, determined.
- **Health Notes**: Common health issues include epilepsy, hip dysplasia, hypothyroidism, and ear infections.
- **Lifespan**: 10-15 years
- **Size**: Small to medium

Beagles are known for their excellent sense of smell and tracking instinct, making them popular as hunting dogs and family pets.


[Agent]: The typical lifespan of a Beagle is **10 to 15 years**. In terms of size, Beagles are classified as **small to medium** dogs. They generally weigh between **20 to 30 pounds** and stand about **13 to 15 inches** tall at the shoulder.


[Agent]: Here are some specific diet recommendations for Beagles:

1. **High-Quality Dog Food**: Choose a d

## 7 · Monitoring — simulated metrics visualization

The charts below simulate what Grafana shows in production.
Run a few agent turns first (cells 5–6), then execute this cell
to read the live Prometheus counters and histograms.


In [7]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend, safe in all environments
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from agent.monitoring.metrics import (
    AGENT_TURN_LATENCY,
    AGENT_TURNS_PER_SESSION,
    AGENT_TOOL_CALLS_TOTAL,
    AGENT_SESSIONS_TOTAL,
)

# ── Helper: extract histogram bucket data ──────────────────────────────────
def _histogram_samples(histogram):
    """Return (bucket_upper_bounds, counts) from a Prometheus Histogram."""
    bounds, counts = [], []
    for sample in histogram.collect()[0].samples:
        if sample.name.endswith("_bucket") and sample.labels.get("le") != "+Inf":
            bounds.append(float(sample.labels["le"]))
            counts.append(sample.value)
    # Convert cumulative counts to per-bucket counts
    per_bucket = [counts[0]] + [counts[i] - counts[i-1] for i in range(1, len(counts))]
    return bounds, per_bucket

# ── Helper: extract counter label data ────────────────────────────────────
def _counter_by_label(counter, label_name):
    """Return {label_value: count} dict from a labelled Prometheus Counter."""
    data = {}
    for sample in counter.collect()[0].samples:
        if not sample.name.endswith("_total"):
            continue
        key = sample.labels.get(label_name, "unknown")
        data[key] = sample.value
    return data

# ── Inject simulated data if metrics are all zero (no live agent run) ──────
def _inject_demo_data():
    """Seed metrics with realistic demo values for portfolio presentation."""
    import random
    random.seed(42)
    latencies = [random.uniform(1.5, 8.0) for _ in range(12)]
    for lat in latencies:
        AGENT_TURN_LATENCY.observe(lat)
    for turns in [2, 3, 2, 4, 3, 5, 2, 3]:
        AGENT_TURNS_PER_SESSION.observe(turns)
    tools_demo = {"cv_predict": 12, "db_query": 10, "web_search": 7, "memory_search": 5}
    for tool, n in tools_demo.items():
        for _ in range(n):
            AGENT_TOOL_CALLS_TOTAL.labels(tool_name=tool).inc()
    for _ in range(8):
        AGENT_SESSIONS_TOTAL.inc()

# Check if metrics have real data; if not, inject demo values
latency_count = next(
    (s.value for s in AGENT_TURN_LATENCY.collect()[0].samples if s.name.endswith("_count")),
    0,
)
if latency_count == 0:
    print("No live data detected — injecting demo values for visualization.")
    _inject_demo_data()

# Register sessions from cells 5–6 using real turn counts
from agent.monitoring.metrics import record_session
from langchain_core.messages import HumanMessage as _HM

if "result_single" in dir():
    record_session(turn_count=result_single["turn_count"])
if "session_ref" in dir():
    turns = sum(1 for m in session_ref.history if isinstance(m, _HM))
    record_session(turn_count=turns)

# ── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Agent Monitoring Dashboard", fontsize=15, fontweight="bold", y=1.02)

PALETTE = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

# Chart 1 — Turn latency histogram
ax = axes[0]
bounds, per_bucket = _histogram_samples(AGENT_TURN_LATENCY)
labels = [f"≤{b}s" for b in bounds]
bars = ax.bar(labels, per_bucket, color=PALETTE[0], edgecolor="white", linewidth=0.5)
ax.set_title("Agent Turn Latency", fontweight="bold")
ax.set_xlabel("Latency bucket")
ax.set_ylabel("Number of turns")
ax.tick_params(axis="x", rotation=35)
for bar, val in zip(bars, per_bucket):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                str(int(val)), ha="center", va="bottom", fontsize=9)

# Chart 2 — Tool calls breakdown (horizontal bar)
ax = axes[1]
tool_data = _counter_by_label(AGENT_TOOL_CALLS_TOTAL, "tool_name")
if tool_data:
    tools_sorted = sorted(tool_data.items(), key=lambda x: x[1], reverse=True)
    tool_names = [t[0] for t in tools_sorted]
    tool_counts = [t[1] for t in tools_sorted]
    colors = PALETTE[:len(tool_names)]
    hbars = ax.barh(tool_names, tool_counts, color=colors, edgecolor="white")
    ax.set_title("Tool Calls by Type", fontweight="bold")
    ax.set_xlabel("Number of calls")
    for hbar, val in zip(hbars, tool_counts):
        ax.text(val + 0.1, hbar.get_y() + hbar.get_height() / 2,
                str(int(val)), va="center", fontsize=9)
else:
    ax.text(0.5, 0.5, "No tool call data", ha="center", va="center",
            transform=ax.transAxes, color="grey")
    ax.set_title("Tool Calls by Type", fontweight="bold")

# Chart 3 — Turns per session histogram
ax = axes[2]
bounds2, per_bucket2 = _histogram_samples(AGENT_TURNS_PER_SESSION)
labels2 = [f"≤{int(b)}" for b in bounds2]
bars2 = ax.bar(labels2, per_bucket2, color=PALETTE[1], edgecolor="white", linewidth=0.5)
ax.set_title("Turns per Session", fontweight="bold")
ax.set_xlabel("Turn bucket")
ax.set_ylabel("Sessions")
for bar, val in zip(bars2, per_bucket2):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                str(int(val)), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("monitoring_dashboard.png", dpi=150, bbox_inches="tight")
from IPython.display import display
display(fig)
print("Chart saved to monitoring_dashboard.png")

# ── Summary stats ──────────────────────────────────────────────────────────
total_sessions = next(
    (s.value for s in AGENT_SESSIONS_TOTAL.collect()[0].samples
     if s.name.endswith("_total")), 0,
)
total_calls = sum(_counter_by_label(AGENT_TOOL_CALLS_TOTAL, "tool_name").values())
print(f"\nTotal sessions  : {int(total_sessions)}")
print(f"Total tool calls: {int(total_calls)}")


<Figure size 1600x500 with 3 Axes>

Chart saved to monitoring_dashboard.png

Total sessions  : 2
Total tool calls: 7


## 8 · System architecture diagram

Text-based overview of the full pipeline (no external dependencies needed).


In [8]:
ARCHITECTURE = """
╔══════════════════════════════════════════════════════════════════════╗
║            AGENTIC VISION CLOUD ORCHESTRATOR — Architecture          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║   User input (image URL / question)                                  ║
║        │                                                             ║
║        ▼                                                             ║
║  ┌─────────────┐                                                     ║
║  │  LangGraph  │  ReAct loop (agent → tools → extract_breed → agent) ║
║  │   Agent     │  GPT-4o-mini · SystemPrompt · MAX_TURNS=10          ║
║  └──────┬──────┘                                                     ║
║         │  tool_calls                                                ║
║    ┌────┴──────────────────────────────────┐                         ║
║    │                                       │                         ║
║    ▼                                       ▼                         ║
║  ┌──────────────┐                ┌─────────────────┐                 ║
║  │  cv_predict  │                │    db_query     │                 ║
║  │  (cv_tool)   │                │   (db_tool)     │                 ║
║  └──────┬───────┘                └────────┬────────┘                 ║
║         │ HTTP POST /predict              │ in-memory lookup         ║
║         ▼                                │                           ║
║  ┌──────────────┐          ┌─────────────┴──────────┐                ║
║  │  CV Service  │          │      web_search        │                ║
║  │  (FastAPI +  │          │   (DuckDuckGo ddgs)    │                ║
║  │  ResNet50)   │          └────────────────────────┘                ║
║  │  Docker      │                                                    ║
║  └──────────────┘          ┌──────────────────────────┐              ║
║                            │  memory_search /         │              ║
║  CV model:                 │  breed_semantic_search   │              ║
║  ResNet50 90.1% acc        │  (Pinecone vector DB)    │              ║
║                            └──────────────────────────┘              ║
║                                                                      ║
║  Monitoring: Prometheus metrics → Grafana dashboard                  ║
║  (agent_turn_latency · agent_tool_calls_total · agent_sessions)      ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(ARCHITECTURE)



╔══════════════════════════════════════════════════════════════════════╗
║            AGENTIC VISION CLOUD ORCHESTRATOR — Architecture          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║   User input (image URL / question)                                  ║
║        │                                                             ║
║        ▼                                                             ║
║  ┌─────────────┐                                                     ║
║  │  LangGraph  │  ReAct loop (agent → tools → extract_breed → agent) ║
║  │   Agent     │  GPT-4o-mini · SystemPrompt · MAX_TURNS=10          ║
║  └──────┬──────┘                                                     ║
║         │  tool_calls                                                ║
║    ┌────┴──────────────────────────────────┐                         ║
║    │                                       │    

## 9 · Offline demo (no Docker required)

This cell runs the agent with mocked CV responses so you can explore
the reasoning pipeline without the `cv_service` container running.
Useful for portfolio demos or CI environments.


In [9]:
from unittest.mock import MagicMock, patch
import json

# Mock httpx so cv_predict returns a fake Beagle prediction
FAKE_CV_RESPONSE = {
    "breed": "Beagle",
    "confidence": 0.91,
    "top5": [
        {"breed": "Beagle",           "confidence": 0.91},
        {"breed": "English Foxhound", "confidence": 0.05},
        {"breed": "Harrier",          "confidence": 0.02},
        {"breed": "Basset Hound",     "confidence": 0.01},
        {"breed": "Bloodhound",       "confidence": 0.01},
    ],
}

mock_resp = MagicMock()
mock_resp.status_code = 200
mock_resp.json.return_value = FAKE_CV_RESPONSE
mock_resp.raise_for_status = lambda: None

mock_client = MagicMock()
mock_client.post.return_value = mock_resp
mock_client.__enter__ = lambda s: mock_client
mock_client.__exit__ = MagicMock(return_value=False)

IMAGE_URL = "https://images.dog.ceo/breeds/beagle/n02088364_10108.jpg"

with patch("httpx.Client", return_value=mock_client):
    from agent.runner import run_agent
    from agent.tools.cv_tool import cv_predict
    from agent.tools.db_tool import db_query

    result = run_agent(
        user_input=f"What breed is in this image? {IMAGE_URL}",
        tools=[cv_predict, db_query],
    )

print("=== Offline demo result ===")
print(result["messages"][-1].content)
print(f"\nBreed identified: {result['breed_identified']}")
print(f"Turns used      : {result['turn_count']}")


=== Offline demo result ===
The breed in the image is a **Beagle** with a confidence score of 91%. 

### Beagle Overview:
- **Type**: Dog
- **Description**: Compact scent hound with floppy ears and a merry temperament.
- **Origin**: United Kingdom
- **Temperament**: Curious, friendly, merry, determined.
- **Health Notes**: Common health issues include epilepsy, hip dysplasia, hypothyroidism, and ear infections.
- **Lifespan**: 10-15 years
- **Size**: Small to medium

Beagles are known for their friendly nature and are often used as scent detection dogs due to their excellent sense of smell.

Breed identified: Beagle
Turns used      : 3


## Summary

| Capability | Implementation | Status |
|------------|---------------|--------|
| Computer Vision | ResNet50 fine-tuned on Oxford Pets (90.1% acc) | ✅ Project "llm-cv-finetuning-pipeline " |
| CV deploy as API | FastAPI + Docker (`cv_service`) | ✅ Phase 2–3 |
| Agentic orchestration | LangGraph ReAct loop | ✅ Phase 4 |
| Tool use | cv_predict · db_query · web_search · memory | ✅ Phase 5 |
| Vector memory | Pinecone embeddings (OpenAI text-embedding-3-small) | ✅ Phase 6 |
| Monitoring | Prometheus + Grafana | ✅ Phase 7 |
| End-to-end testing | 86 tests, 0 regressions | ✅ Phase 8 |
| Interactive demo | This notebook | ✅ Phase 9 |

### Key portfolio talking points

- **End-to-end ownership**: trained the CV model (Project "llm-cv-finetuning-pipeline") → deployed it as a
  microservice → integrated it as a tool inside a LangGraph agent.
- **ReAct pattern**: the agent autonomously decides when to call which tool
  based on the user query, without hardcoded branching.
- **Production-grade**: Docker, Prometheus metrics, Pinecone memory,
  graceful error handling, 86-test suite.
